# Local fruit identification and ripeness evaluation

Evaluate the same Python backend that the future application will import: local **Qwen3-VL 4B through Ollama**. No frontend is needed.

The 20 fixtures have frozen development/held-out splits and image checksums. Labels were assigned by the implementation assistant from visible appearance **before evaluation**, not by an independent food expert. This is a small functionality check, not a calibrated accuracy or food-safety benchmark.

**Hypotheses:** clear fruit types should be recognized; maturity should follow fruit-specific cues; ambiguous maturity should allow uncertainty; multiple fruits and unreadable inputs should be rejected. Parsing success and predictive correctness are measured separately.

**Wrap-up:** the final run completed all 13 development cases and 4 of 7 held-out cases. Evaluation stopped at the user’s request. The other 3 held-out cases are untested; no results are inferred for them.

In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
from collections import Counter
from datetime import datetime, timezone
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fruit_backend").is_dir())
from fruit_backend import analyze_image
from fruit_backend.evaluation import load_cases, run_evaluation, summarize
from fruit_backend.prompts import PROMPT_VERSION

MANIFEST = ROOT / "evaluation/manifest.json"
cases = load_cases(MANIFEST)
RUN_LIVE = os.environ.get("FRUIT_EVAL_LIVE", "0") == "1"
LIVE_SPLIT = "development"  # Keep held-out examples separate from prompt development.
print({name: importlib.metadata.version(name) for name in ["ollama", "Pillow", "pydantic"]})
print({"prompt": PROMPT_VERSION, "cases": len(cases), "live_inference": RUN_LIVE})

{'ollama': '0.6.3', 'Pillow': '13.0.0', 'pydantic': '2.14.4'}
{'prompt': 'fruit-v5', 'cases': 20, 'live_inference': False}


## Frozen references and provenance

Most fixtures are openly licensed photographs; one is a synthetic checkerboard and one is an intentionally degraded photograph. Cropped derivatives share a source group with their originals and never cross splits. Full image licenses and modification details are in `evaluation/ATTRIBUTION.md`.

Inspect the manifest's `label_note` before treating a ripeness disagreement as a definitive model error. Expected `uncertain` cases are evaluated separately from assessable ripeness.

In [2]:
def markdown_table(headers, rows):
    def escape(value):
        return str(value).replace("|", "/").replace("\n", " ")
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(escape(v) for v in row) + " |" for row in rows]
    return Markdown("\n".join(lines))

display(markdown_table(["Case", "Split", "Fruit", "Ripeness", "Scene status"], [
    [c.id, c.split, c.expected_fruit, c.expected_ripeness, c.expected_status] for c in cases
]))

| Case | Split | Fruit | Ripeness | Scene status |
| --- | --- | --- | --- | --- |
| banana-yellow | development | banana | ripe | assessed |
| banana-dark | development | banana | overripe | assessed |
| banana-damaged | held_out | banana | uncertain | assessed |
| apple-red | development | apple | ripe | assessed |
| apple-green | development | apple | ripe | assessed |
| apple-damaged | held_out | apple | uncertain | assessed |
| orange-whole | development | orange | ripe | assessed |
| orange-green | development | orange | unripe | assessed |
| orange-mold | held_out | orange | uncertain | assessed |
| strawberry-whole | development | strawberry | ripe | assessed |
| strawberry-half | development | strawberry | ripe | assessed |
| strawberry-pale | held_out | strawberry | unripe | assessed |
| avocado-cut | development | avocado | ripe | assessed |
| avocado-half | development | avocado | ripe | assessed |
| avocado-tree | held_out | avocado | uncertain | assessed |
| multiple-bananas | development | None | uncertain | multiple_fruits |
| nonfruit-checkerboard | development | None | uncertain | no_fruit |
| unreadable-banana | development | None | uncertain | unclear_image |
| decomposed-apple | held_out | apple | uncertain | assessed |
| multiple-citrus-mold | held_out | None | uncertain | multiple_fruits |

## Run or inspect saved predictions

By default this notebook **replays saved real-model evaluation results**; it does not make hidden model calls. Set `FRUIT_EVAL_LIVE=1` before executing to run the development split again through the production backend. A timestamped output file preserves earlier reports. Ollama must be running and `qwen3-vl:4b` installed for live execution.

The public one-image entry point is `analyze_image(Path("photo.jpg").read_bytes())`. Batch evaluation reuses a `FruitAnalyzer` instance but starts a fresh chat for every image.

In [3]:
if RUN_LIVE:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    report = run_evaluation(MANIFEST, ROOT / f"output/evaluation/notebook-{stamp}.json", split=LIVE_SPLIT)
else:
    report = json.loads((ROOT / "output/evaluation/final-evaluation.json").read_text())

assert report["complete"] or report.get("stopped_early", False), "Unexplained incomplete report."
if report.get("stopped_early", False):
    print(f"Stopped at user request: {len(report['records'])}/{report['planned_cases']} cases completed; remaining cases were not evaluated.")
assert report["manifest_sha256"] == hashlib.sha256(MANIFEST.read_bytes()).hexdigest()
assert report["prompt_version"] == PROMPT_VERSION, "Saved results use a different prompt; run a fresh evaluation."
records = report["records"]
print({"mode": "live inference" if RUN_LIVE else "saved real-model results", "prompt": report["prompt_version"], "model_digest": report["model_digest"], "evaluated": len(records)})

Stopped at user request: 17/20 cases completed; remaining cases were not evaluated.
{'mode': 'saved real-model results', 'prompt': 'fruit-v5', 'model_digest': 'a1b29c246319cdcb1f3842f36d29f198f737c0cf516ba1bcba34b1b974e87980', 'evaluated': 17}


## Results

Fruit accuracy includes all references with an identifiable fruit. Ripeness agreement excludes reference-uncertain images, but counts model abstentions and operational errors as misses on assessable examples. A maturity label for the wrong fruit is not a correct result. Spoilage and scene rejection are separate measures.

All condition metrics also require the correct fruit identity. Timing summaries describe successfully completed inferences; operational failures remain counted in the accuracy metrics.

In [4]:
metric_rows = []
for split in ["development", "held_out"]:
    subset = [r for r in records if r["split"] == split]
    if not subset:
        continue
    summary = summarize(subset)
    for metric in ["fruit_identification", "ripeness_on_assessable_images", "uncertainty_on_ambiguous_ripeness", "visible_spoilage", "scene_rejection"]:
        value = summary[metric]
        metric_rows.append([split, metric, value["correct"], value["total"], value["rate"]])
    print(split, {key: summary[key] for key in ["cases", "operational_errors", "invalid_outputs", "abstentions_on_assessable_ripeness"]})
display(markdown_table(["Split", "Metric", "Correct", "Total", "Rate"], metric_rows))

development {'cases': 13, 'operational_errors': 0, 'invalid_outputs': 0, 'abstentions_on_assessable_ripeness': 0}
held_out {'cases': 4, 'operational_errors': 0, 'invalid_outputs': 0, 'abstentions_on_assessable_ripeness': 0}


| Split | Metric | Correct | Total | Rate |
| --- | --- | --- | --- | --- |
| development | fruit_identification | 9 | 10 | 0.9 |
| development | ripeness_on_assessable_images | 8 | 10 | 0.8 |
| development | uncertainty_on_ambiguous_ripeness | 0 | 0 | None |
| development | visible_spoilage | 7 | 10 | 0.7 |
| development | scene_rejection | 2 | 3 | 0.6667 |
| held_out | fruit_identification | 3 | 4 | 0.75 |
| held_out | ripeness_on_assessable_images | 1 | 1 | 1.0 |
| held_out | uncertainty_on_ambiguous_ripeness | 0 | 3 | 0.0 |
| held_out | visible_spoilage | 1 | 4 | 0.25 |
| held_out | scene_rejection | 0 | 0 | None |

In [5]:
prediction_rows = []
for row in records:
    predicted = (row.get("result") or {}).get("assessment", {})
    prediction_rows.append([row["id"], predicted.get("status", "ERROR"), predicted.get("fruit_name"),
        row["expected_ripeness"], predicted.get("ripeness"), predicted.get("visible_spoilage"),
        (row.get("error") or {}).get("code", "")])
display(markdown_table(["Case", "Predicted status", "Predicted fruit", "Expected ripeness", "Predicted ripeness", "Visible spoilage", "Error"], prediction_rows))

| Case | Predicted status | Predicted fruit | Expected ripeness | Predicted ripeness | Visible spoilage | Error |
| --- | --- | --- | --- | --- | --- | --- |
| banana-yellow | assessed | banana | ripe | ripe | none_observed |  |
| banana-dark | assessed | banana | overripe | ripe | none_observed |  |
| apple-red | assessed | apple | ripe | ripe | none_observed |  |
| apple-green | assessed | apple | ripe | ripe | none_observed |  |
| orange-whole | assessed | orange | ripe | ripe | none_observed |  |
| orange-green | assessed | mango | unripe | unripe | none_observed |  |
| strawberry-whole | assessed | strawberry | ripe | ripe | none_observed |  |
| strawberry-half | assessed | strawberry | ripe | ripe | possible |  |
| avocado-cut | assessed | avocado | ripe | ripe | none_observed |  |
| avocado-half | assessed | avocado | ripe | ripe | none_observed |  |
| multiple-bananas | multiple_fruits | None | uncertain | uncertain | uncertain |  |
| nonfruit-checkerboard | no_fruit | None | uncertain | uncertain | uncertain |  |
| unreadable-banana | no_fruit | None | uncertain | uncertain | uncertain |  |
| banana-damaged | assessed | banana | uncertain | ripe | none_observed |  |
| apple-damaged | assessed | apple | uncertain | ripe | none_observed |  |
| orange-mold | assessed | lime | uncertain | unripe | clear |  |
| strawberry-pale | assessed | strawberry | unripe | unripe | none_observed |  |

## Latency and disagreements

Report the first request and subsequent median separately. These are request-order measurements, not a guaranteed cold/warm distinction: Ollama may already have a model or prompt cached. Per-request model load duration is saved in the raw reports. The initial smoke test separately records model startup time.

A disagreement with an uncertain reference can reveal overconfidence even when fruit identification is correct.

In [6]:
timing = summarize(records)
print({key: timing[key] for key in ["first_inference_seconds", "subsequent_median_seconds", "max_inference_seconds"]})
smoke = json.loads((ROOT / "output/evaluation/baseline-banana-v1.json").read_text())
print("Initial smoke request:", {key: smoke["metadata"][key] for key in ["inference_seconds", "model_load_seconds"]})

failures = []
for row in records:
    predicted = (row.get("result") or {}).get("assessment", {})
    mismatch = row.get("error") is not None or predicted.get("status") != row["expected_status"]
    if row["expected_status"] == "assessed":
        mismatch |= predicted.get("fruit_name") not in [row["expected_fruit"], *row["fruit_aliases"]]
        mismatch |= predicted.get("ripeness") != row["expected_ripeness"]
        mismatch |= predicted.get("visible_spoilage") != row["expected_visible_spoilage"]
    if mismatch:
        failures.append([row["id"], row["label_note"], predicted.get("explanation", str(row.get("error")))])
display(markdown_table(["Disagreement", "Reference rationale", "Model explanation"], failures))

{'first_inference_seconds': 8.456, 'subsequent_median_seconds': 11.2851, 'max_inference_seconds': 104.2538}
Initial smoke request: {'inference_seconds': 17.899, 'model_load_seconds': 2.831350208}


| Disagreement | Reference rationale | Model explanation |
| --- | --- | --- |
| banana-dark | Extensive darkening/shriveling; no claim about internal condition or edibility. | Bananas are typically ripe when dark brown; dried stem is common for ripe bananas. No mold or decomposition indicates no spoilage. |
| orange-green | Small green fruit on tree; source identifies immature orange, but color alone is not universally diagnostic. | Green color indicates unripe state; no visible spoilage. |
| strawberry-half | One cut half with red exterior and light central flesh. | Ripe due to bright red color; brown discoloration in the central cavity may indicate possible spoilage. |
| unreadable-banana | Synthetic severe loss of image detail; kept in same split as original. | The image is a blurred gradient of light colors with no discernible fruit. No fruit is visible or identifiable. |
| banana-damaged | Long damaged peel and yellow color do not cleanly distinguish bruising from advanced ripening. | Yellow color indicates ripeness; minor brown spots are not extensive enough to suggest overripeness or spoilage. |
| apple-damaged | Large cavity and discoloration; visible damage is clearer than ripeness. | Ripeness is indicated by the yellow-red color, which is typical for ripe apples. The large bite mark is a natural feature, not spoilage. No mold or decay is visible. |
| orange-mold | Visible mold on peeled citrus; peel pieces belong to the single fruit. Spoilage does not establish maturity. | The green fruit is a lime, typically unripe when green. Visible mold on the fruit indicates clear spoilage. Rind pieces exhibit brown discoloration and separation, consistent with decay. |

## Interpretation and next steps

- These labels describe visible appearance. Several maturity labels remain provisional, especially for apples, citrus, and avocados.
- The public photos may have been present in model training data. This small set cannot establish performance on new phone photos.
- One strawberry pair comes from closely related photography and is kept together in the development split. Crops and synthetic controls are explicitly documented.
- Do not tune against held-out failures and continue calling those results untouched hold-out accuracy. Collect fresh examples for the next revision.
- Before frontend work, test the actual fruits and lighting intended for the hackathon demo. Preserve every error and uncertain response.
- Uploaded user images are not persisted by the backend. Only this explicitly bundled evaluation set and its reports are stored.